# Charting the Grind

A project for the **Information Visualization course 2025/26** about understanding **personal study habits** through the visual exploration of a self-tracked study log, collected over roughly the last month of study sessions.

I started keeping this log because I wanted a clearer picture of what my study life actually looks like, beyond the vague feeling of "I studied a lot" or "this afternoon was not very productive". By looking at my sessions visually, I want to notice patterns I would probably miss otherwise: which topics take most of my time, when I seem to work better, which places help me focus and which kinds of activities feel more or less productive.

## Preparing the data

Before creating the visualizations, the raw study log needs to be cleaned and enriched. The CSV already contains the basic information about each study session, but some columns are still difficult to analyze directly. Primarily, start and finish times are written as text and the actual duration of each session is not explicitly available.  
In this phase, I **validate the dataset structure** and prepare it so that it can be used more easily for charts and interpretation.

First, I prepare the environment by importing the **libraries** I need and declaring all the **constants** used throughout the notebook.


In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

In [2]:
# the input data csv file path
DATA_CSV = Path("data/study_sessions.csv")

# the output directory for the generated charts
CHARTS_DIR = Path("charts")

# the columns that should be treated as categorical
CATEGORICAL_COLS = ["topic", "activity_type", "location"]

# the mapping of time of day bins for categorization
TIME_OF_DAY_CUTS = {
    "bins": [8, 13.5, 18.5, 24],
    "labels": ["morning", "afternoon", "evening"],
}

Then, I read the data from the CSV file into a **dataframe** and check its structure to make sure it contains all the expected columns, that they are in the right format, and that there are **no missing values** or other issues that could affect the analysis.

In [3]:
df = pd.read_csv(DATA_CSV)

In [4]:
# check the structure of the dataframe, especially the data types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   date                100 non-null    str  
 1   start               100 non-null    str  
 2   finish              100 non-null    str  
 3   topic               100 non-null    str  
 4   activity_type       100 non-null    str  
 5   productivity_score  100 non-null    int64
 6   location            100 non-null    str  
 7   description         100 non-null    str  
dtypes: int64(1), str(7)
memory usage: 6.4 KB


I now need to process the data a little to make it more suitable for analysis.

First, let's transform the **start** and **finish** times from text to datetime format and calculate the **duration** of each session in hours and minutes. This will allow us to analyze how long each session lasted and to create time-based visualizations later on.

In [5]:
# combine date + time into datetime columns
df["start_dt"] = pd.to_datetime(df["date"] + " " + df["start"], dayfirst=True)
df["finish_dt"] = pd.to_datetime(df["date"] + " " + df["finish"], dayfirst=True)

# calculate duration as a time delta
df["duration"] = df["finish_dt"] - df["start_dt"]

# convert duration to hours as a float
df["duration_hours"] = df["duration"].dt.total_seconds() / 3600

# convert duration to minutes as an integer
df["duration_minutes"] = (df["duration"].dt.total_seconds() / 60).astype(int)

# print the start, finish and duration columns to check the calculations
print(df[["start_dt", "finish_dt", "duration_hours", "duration_minutes"]])

              start_dt           finish_dt  duration_hours  duration_minutes
0  2026-04-13 10:00:00 2026-04-13 12:30:00             2.5               150
1  2026-04-14 09:30:00 2026-04-14 12:00:00             2.5               150
2  2026-04-14 15:30:00 2026-04-14 17:00:00             1.5                90
3  2026-04-14 17:00:00 2026-04-14 17:30:00             0.5                30
4  2026-04-14 18:30:00 2026-04-14 21:00:00             2.5               150
..                 ...                 ...             ...               ...
95 2026-05-19 21:00:00 2026-05-19 23:30:00             2.5               150
96 2026-05-20 09:00:00 2026-05-20 11:00:00             2.0               120
97 2026-05-20 11:00:00 2026-05-20 12:00:00             1.0                60
98 2026-05-20 16:00:00 2026-05-20 18:00:00             2.0               120
99 2026-05-20 18:00:00 2026-05-20 22:00:00             4.0               240

[100 rows x 4 columns]


Then, let's infer the **time of the day** for each session based on the start time. This will help us understand when I tend to study and whether there are any patterns in my productivity throughout the day.

In [6]:
# infer time of day from the start time using the defined bins and labels
df["time_of_day"] = pd.cut(
    df["start_dt"].dt.hour + df["start_dt"].dt.minute / 60,
    **TIME_OF_DAY_CUTS,
    right=False,
)

# print the start, finish and time of day columns to check the categorization
print(df[["start_dt", "finish_dt", "time_of_day"]])

              start_dt           finish_dt time_of_day
0  2026-04-13 10:00:00 2026-04-13 12:30:00     morning
1  2026-04-14 09:30:00 2026-04-14 12:00:00     morning
2  2026-04-14 15:30:00 2026-04-14 17:00:00   afternoon
3  2026-04-14 17:00:00 2026-04-14 17:30:00   afternoon
4  2026-04-14 18:30:00 2026-04-14 21:00:00     evening
..                 ...                 ...         ...
95 2026-05-19 21:00:00 2026-05-19 23:30:00     evening
96 2026-05-20 09:00:00 2026-05-20 11:00:00     morning
97 2026-05-20 11:00:00 2026-05-20 12:00:00     morning
98 2026-05-20 16:00:00 2026-05-20 18:00:00   afternoon
99 2026-05-20 18:00:00 2026-05-20 22:00:00   afternoon

[100 rows x 3 columns]


Finally, let's convert some of the columns that I know should be categorical to the appropriate data type for easier analysis.

In [7]:
# convert columns marked as categorical to category data type
df[CATEGORICAL_COLS] = df[CATEGORICAL_COLS].astype("category")

# print the unique values in categorical columns
for col in CATEGORICAL_COLS:
    unique_values = df[col].unique().tolist()
    unique_values.sort()
    print(f"Unique values in '{col}': {unique_values}")

Unique values in 'topic': ['Digital Scholarly Editing', 'Freelance Work', 'Information Visualization', 'Knowledge Representation', 'Open Science', 'Semantic Digital Libraries']
Unique values in 'activity_type': ['brainstorming', 'programming', 'reading', 'reviewing', 'writing']
Unique values in 'location': ['32', '34', '36', 'bigiavi', 'home', 'paleotti']


In [8]:
# final preflight check of the dataframe structure after all transformations
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   date                100 non-null    str            
 1   start               100 non-null    str            
 2   finish              100 non-null    str            
 3   topic               100 non-null    category       
 4   activity_type       100 non-null    category       
 5   productivity_score  100 non-null    int64          
 6   location            100 non-null    category       
 7   description         100 non-null    str            
 8   start_dt            100 non-null    datetime64[us] 
 9   finish_dt           100 non-null    datetime64[us] 
 10  duration            100 non-null    timedelta64[us]
 11  duration_hours      100 non-null    float64        
 12  duration_minutes    100 non-null    int64          
 13  time_of_day         100 non-null    category   

## Visualizing the data

After preparing the dataframe and adding the columns needed for analysis, I can now start **turning the data into visualizations**.  
In this phase, the goal is to move from the raw list of study sessions to charts that make my **habits understandable** at a glance, **highlight meaningful patterns** and possibly help me reflect on **how to organize future study** time more consciously.


### Visualization 1: Total study time per topic

The most obvious information I want to visualize is how much time I have spent on **each topic** compared to the others, and whether this confirms my **intuitive feeling** of where my study time has gone. The data is not complete: some topics had already been started, or were almost finished, when I began keeping the log, while others will continue after the project has been delivered. Still, this visualization is useful as a quick **vibe check**, because it gives me an immediate overview of the relative weight of each topic inside the recorded period.

To visualize this, I will **group the sessions** by `topic` and sum the `duration_hours` column, which was calculated from the `start` and `finish` times during the data preparation phase. The result will be a simple but effective **bar chart**, where each bar represents a topic and its total recorded study time. This makes it easy to compare topics at a glance and see which ones absorbed most of my attention during the logging period.

In [9]:
# group data by topic and calculate total study time using the duration_hours, then sort in ascending order
topic_hours = (
    df.groupby("topic", as_index=False)["duration_hours"]
    .sum()
    .sort_values("duration_hours", ascending=True)
)

# print the new dataframe for a quick check of the results
print(topic_hours)

                        topic  duration_hours
5  Semantic Digital Libraries            8.50
0   Digital Scholarly Editing           12.50
1              Freelance Work           17.75
2   Information Visualization           24.25
3    Knowledge Representation           58.50
4                Open Science           73.25


In [10]:
# create a horizontal bar chart and customize it with titles, labels and colors
fig = px.bar(
    topic_hours,
    x="duration_hours",
    y="topic",
    orientation="h",
    title="Total study time per Topic",
    labels={
        "duration_hours": "Total study time (hours)",
        "topic": "Topic"
    },
    text="duration_hours",
    color_discrete_sequence=["#4C78A8"]
)

# show the total hours as text labels alongside the bars with a single decimal place
fig.update_traces(
    texttemplate="%{text:.1f} h",
    textposition="outside"
)

# compute the maximum hours value for x-axis padding for better visualization of the text labels
max_hours = topic_hours["duration_hours"].max()

# customize the layout of the figure for better aesthetics and readability
fig.update_layout(
    height=600,
    showlegend=False,
    title_font=dict(size=25),
    xaxis_title="Total study time (hours)",
    yaxis_title="Topic",
    margin=dict(l=220, r=130, t=100, b=140),
    xaxis=dict(
        range=[0, max_hours * 1.15]
    ),
    yaxis=dict(
        title=dict(
            text="Topic",
            standoff=25
        )
    ),
    annotations=[
        dict(
            text="This chart compares the total amount of recorded study time for each topic: "
                 "it gives a quick overview of which subjects took up "
                 "most of my<br>attention during the logging period.",
            x=-0.08,
            y=-0.3,
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left",
            font=dict(size=11),
        )
    ],
)

# save the figure as a static high-resolution image to the charts directory
fig.write_image(
    CHARTS_DIR / "total_study_time_per_topic.jpg",
    width=1200,
    height=600,
    scale=2
)

# display the figure in the notebook
fig.show()

The chart confirmed most of my expectations: the topics that felt more present in my daily routine are also the ones with the highest number of recorded study hours.

- **Open Science** requires most of my time and attention because I need to write complex Python code to manipulate huge amount of data for the exam project. I've written and rewritten the code several times, and I still have a lot of work to do, so it's not surprising that this topic takes up most of my study time.

- **Knowledge Representation** is the second topic in terms of hours, and this also makes sense because it was the exam I started working on when I began keeping the log. Moreover, while I passed the exam more than a week before the end of the logging period, I still spent some time on it to prepare a new release of the ontology we worked on in order to publish it on Zenodo.

- **Information Visualization** is the third topic and it's in the right place. It doesn't have the frustrating complexity of Open Science, or the obsessive perfectionism of Knowledge Representation, but it skyrocketed in the last week of the logging period because I had to prepare the project for submission and I wanted to make it as good as possible.

- **Freelance Work** and **Semantic Digital Libraries** are also quite where I expected. I stopped doing freelance work when the exam period started (and also because I didn't like the company I was working for anymore 🤭), while the SDL exam was a few days after the logging started.

- **Digital Scholarly Editing** however is surprisingly low in terms of hours, since I have the exam the same day as Information Visualization and I expected to have spent more time on it. So I will probably fail the exam 😎.